In [1]:
# imports 
import numpy as np
import torch
import torch.nn as nn
from accelerate import init_empty_weights
from accelerate.utils.modeling import find_tied_parameters, \
get_mixed_precision_context_manager
from accelerate.utils.operations import convert_outputs_to_fp32
from bitsandbytes.nn import Linear8bitLt, Linear4bit, LinearFP4, LinearNF4
from collections import Counter
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer, \
AutoConfig
from transformers.integrations.bitsandbytes import get_keys_to_not_convert
from types import MethodType

/home/ujjwal/projects/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


quantanized model : to reducec their memory footprint , i.e shrink the model size  

model_size_in_mb = (num_parms * (bits_per_parm / 8)) / 1e6

In [2]:
torch.manual_seed(11)
weights = torch.randn(1000) * .07
weights.min(), weights.max()

(tensor(-0.2066), tensor(0.2097))

In [3]:
n_bins = 4
bins = torch.linspace(weights.min(), weights.max(), n_bins+1)
bin_width = bins[1]-bins[0]
bins, bin_width

(tensor([-0.2066, -0.1026,  0.0015,  0.1056,  0.2097]), tensor(0.1041))

In [4]:
def quantize(weights, n_bits=8):
    assert n_bits <= 16, "Using more bits may result in slow execution and/or crashing."
    n_bins = 2**n_bits
    bins = torch.linspace(weights.min(), weights.max(), n_bins+1)
    first_bin = bins[0]
    bin_width = bins[1]-bins[0]
    bin_indexes = ((weights.view(-1, 1) > bins).to(torch.int).argmin(dim=1) * 1)
    return bin_indexes, bin_width, first_bin

In [5]:
def dequantize (bin_indexes, bin_width, first_bin):
    aprrox_values = bin_indexes * bin_width  + first_bin 
    return aprrox_values

RMSE of several quantization choices:

In [6]:
mse_fn = nn.MSELoss()

In [7]:
for n_bits in [2, 4, 8, 16]:
    res = quantize(weights, n_bits=n_bits)
    approx_values = dequantize(*res)
    print(f'{n_bits}-bit Quantization:')
    print(approx_values[:6])
    print(weights[:6])
    print(mse_fn(approx_values, weights).sqrt())

2-bit Quantization:
tensor([ 0.0015,  0.1056,  0.0015,  0.1056,  0.0015, -0.1026])
tensor([-0.0358,  0.0720, -0.0247,  0.0086, -0.0127, -0.1048])
tensor(0.0591)
4-bit Quantization:
tensor([-0.0245,  0.0796, -0.0245,  0.0275,  0.0015, -0.1026])
tensor([-0.0358,  0.0720, -0.0247,  0.0086, -0.0127, -0.1048])
tensor(0.0146)
8-bit Quantization:
tensor([-0.0343,  0.0731, -0.0245,  0.0096, -0.0115, -0.1042])
tensor([-0.0358,  0.0720, -0.0247,  0.0086, -0.0127, -0.1048])
tensor(0.0009)
16-bit Quantization:
tensor([-0.0359,  0.0718, -0.0248,  0.0085, -0.0128, -0.1049])
tensor([-0.0358,  0.0720, -0.0247,  0.0086, -0.0127, -0.1048])
tensor(0.0001)


# Clearly, the more bits we use, the better the approximation will be.

## loading models

In [8]:
def get_parm_dtypes(iterable, top_k=3):
    return Counter([p.dtype for p in iterable]).most_common(top_k)

In [9]:
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m", device_map='cuda:0')

In [15]:
print(model.get_memory_footprint()/1e6 )

1324.785664


## whole thing takes 1324MB in memory.

In [17]:
get_parm_dtypes(model.parameters())

[(torch.float32, 388)]

###  All of its 388 modules are in torch.float32, and the whole thing takes 1324MB in memory.

In [19]:
state_dict = torch.load('pytorch_model.bin')
get_parm_dtypes(iter(state_dict.values()))

[(torch.float16, 388)]

In [4]:
model = AutoModelForCausalLM.from_pretrained(
"facebook/opt-350m", device_map='cuda:0', dtype=torch.float32)
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

In [87]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m") 
batch= tokenizer(["this is the test"] , return_tensors='pt')


In [88]:
list(batch.items())

[('input_ids', tensor([[   2, 9226,   16,    5, 1296]])),
 ('attention_mask', tensor([[1, 1, 1, 1, 1]]))]

In [89]:
batch['input_ids'].device  , batch['attention_mask'].device

(device(type='cpu'), device(type='cpu'))

## converting to cuda

In [90]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch = {k : v.to(device) for k , v in batch.items()}
batch 

{'input_ids': tensor([[   2, 9226,   16,    5, 1296]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1]], device='cuda:0')}

In [91]:
batch['input_ids'].dtype , batch['attention_mask'].dtype

(torch.int64, torch.int64)

In [5]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
batch = tokenizer(['This is a simple test'], return_tensors='pt')
batch['labels'] = batch['input_ids']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch = {k: v.to(device) for k, v in batch.items()}
out = model(**batch)
out.loss

tensor(3.8001, device='cuda:0', grad_fn=<NllLossBackward0>)

In [ ]:
batch['input_ids'].dtype , batch['attention_mask'].dtype

(torch.int64, torch.int64)

# loss value , 3.8001 

#  half precision modedls (16 bit)

In [101]:
supported = torch.cuda.is_bf16_supported(including_emulation=False)
supported

True

In [103]:
dtype16 = (torch.bfloat16 if supported else torch.float16)
dtype16


torch.bfloat16

In [104]:
model.to(dtype16)
print(model.get_memory_footprint()/1e6, get_parm_dtypes(model.parameters()))

662.392832 [(torch.bfloat16, 388)]


## out model is now half pricision on e , 

# it is taking 662mb in ram , previously it was taking 1324MB

## memory is half  what about loss ?

it’s the very same model and the same
inputs; the only difference is that the weights are torch.float16 now:

In [12]:
batch

{'input_ids': tensor([[   2,  713,   16,   10, 2007, 1296]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0'),
 'labels': tensor([[   2,  713,   16,   10, 2007, 1296]], device='cuda:0')}

In [7]:
## f16
model_fp16= model.to(torch.float16) #f16 changes
out_fp16= model_fp16(**batch)
out_fp16.loss

tensor(3.8010, device='cuda:0', grad_fn=<NllLossBackward0>)

## losss 3.8010

In [17]:
## b16
model_b16= model.to(torch.bfloat16) ## b16  changes
out_b16= model_b16(**batch)
out_fp16.loss

tensor(3.7995, device='cuda:0', grad_fn=<NllLossBackward0>)

## loss 3.7995

#####  b16 has good accuracy i.e loss is smaller then f16

# mixed precision 

We keep the weights and inputs in full precision (FP32)

Before computing (e.g., the forward pass), we cast them to half-precision (FP16 or BF16).

We perform computation in half-precision (FP16 or BF16) for a speedup.

We cast the results back to their original precision (FP32).

In [9]:
class MixedModel(nn.Module):
    def __init__(self, dtype):
        super().__init__()
        self.a = nn.Linear(1000, 1000, dtype=dtype)
        self.b = nn.Linear(1000, 1000, dtype=dtype)
        
    def forward(self, x):
        return self.b(self.a(x))

In [17]:
# create instance of fp32 modedl 
mixed32= MixedModel(torch.float32)
mixed32.to('cuda')

MixedModel(
  (a): Linear(in_features=1000, out_features=1000, bias=True)
  (b): Linear(in_features=1000, out_features=1000, bias=True)
)

In [18]:
%timeit mixed32(torch.randn(1000, 1000, dtype=torch.float32, device='cuda'))

1.35 ms ± 2.57 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### it takes 1.35  miliseconds  compute forward pass on f32 input

## for f16   now

In [15]:
mixed16 = MixedModel(torch.float16) 
mixed16.to('cuda')

MixedModel(
  (a): Linear(in_features=1000, out_features=1000, bias=True)
  (b): Linear(in_features=1000, out_features=1000, bias=True)
)

In [16]:
%timeit mixed16(torch.randn(1000, 1000, dtype=torch.float16, device='cuda'))

437 μs ± 3.29 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## it takes 434 micro seconds , roughly more then  3.11× faster on f16

###  thus mixpricision is good , so we use autocast of hugging face to do this it  will automaitcally 

In [21]:
with torch.autocast(device_type="cuda", dtype= torch.float16):
    result_16= mixed16(torch.randn(1000, 1000, dtype=torch.float32, device= "cuda"))

res_32= result_16.float() 

In [ ]:
autocast_context = torch.autocast(device_type="cuda", dtype=torch.float16)  # create autocast context to run operations in float16 on CUDA

model_forward_func = mixed32.forward.__func__  # extract original unbound forward function from the model

new_forward= autocast_context(model_forward_func)  # wrap original forward function so it runs under autocast (automatic fp16)

mixed32.forward = MethodType(new_forward, mixed32)  # bind the new autocast-enabled forward method back to the model instance

Now, if we call our FP32 model, its (wrapped) forward() method will produce FP16 outputs.

In [26]:
res = mixed32(torch.randn(1000, 1000, dtype=torch.float32, device='cuda'))
res.dtype

torch.float16

Every time we  call mixed32(...) the whole forward pass runs inside the autocast context.

Most expensive ops (matmul, conv, linear, softmax, layer norm, etc.) run in fp16.

The final output of the model also stays in fp16 (very common default behavior when you don't do anything special).

##   now casts its output to back to full precision

In [ ]:
mixed32.forward = MethodType(convert_outputs_to_fp32(mixed32.forward.__func__), mixed32)

res = mixed32(torch.randn(1000, 1000, dtype=torch.float32, device='cuda'))
res.dtype

torch.float16

# BitsAndBytes

In [29]:
bnb_config = BitsAndBytesConfig()
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": false,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": false,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

### most common ,   set either load_in_8bit=True or load_in_4bit=True 

### 8-Bit Quantization

####  "LLM.int8() is a quantization method that doesn’t degrade performance which makes large model inference more

# How LLM.int8 Works: Running Large Models With Almost No Accuracy Loss

- extract t he outliers from the inputs and weights then multiply them in 16 bit 
all other values which might not  loose their values due to .....
are quantalized to 8bit or int8 before dedquantized back to 16-bits

# 8 bit layers

quantanization of each layers occurs when they are set to gpu ...

"In order to quantize a linear layer one should first load the original fp16 / bf16 weights into the Linear8bitLt
module, then call int8_module.to("cuda") to quantize the fp16 weights."

fp_layer.state_dict() These are FP32 tensors. 

because 
In PyTorch, nn.Linear parameters are FP32 tensors by default.


In [13]:
n_in = 10
n_out = 10
fp_layer = nn.Linear(n_in, n_out)
fp_layer.state_dict()

OrderedDict([('weight',
              tensor([[ 0.1221,  0.1194,  0.0393, -0.2798,  0.1914,  0.1671,  0.2771, -0.0392,
                       -0.1663,  0.1416],
                      [ 0.1654,  0.1164, -0.2240, -0.0187, -0.0397, -0.1088,  0.0235, -0.2571,
                        0.0772, -0.1679],
                      [-0.1434, -0.0419,  0.2329,  0.1961,  0.0326, -0.2356, -0.0961,  0.2944,
                       -0.0649, -0.2710],
                      [-0.1147, -0.0216,  0.0824, -0.2227,  0.1101,  0.2807,  0.0139,  0.2451,
                       -0.2678,  0.1882],
                      [ 0.2211,  0.1689, -0.0283,  0.0555, -0.0697,  0.0540,  0.2401,  0.0699,
                        0.3083,  0.3003],
                      [-0.0309, -0.0716,  0.0863, -0.2755,  0.0915,  0.1958,  0.2541, -0.0222,
                       -0.1734,  0.2968],
                      [-0.0846, -0.2036,  0.0659, -0.2901,  0.1971,  0.0968,  0.1876,  0.0645,
                        0.1432, -0.2511],
                 

## we copyed the weight and bais to int8 layers

In [22]:
int8_layer = Linear8bitLt(n_in, n_out, has_fp16_weights=False)
weight_bais= fp_layer.state_dict()
int8_layer.load_state_dict(weight_bais)
int8_layer.state_dict()

OrderedDict([('weight',
              tensor([[ 0.1221,  0.1194,  0.0393, -0.2798,  0.1914,  0.1671,  0.2771, -0.0392,
                       -0.1663,  0.1416],
                      [ 0.1654,  0.1164, -0.2240, -0.0187, -0.0397, -0.1088,  0.0235, -0.2571,
                        0.0772, -0.1679],
                      [-0.1434, -0.0419,  0.2329,  0.1961,  0.0326, -0.2356, -0.0961,  0.2944,
                       -0.0649, -0.2710],
                      [-0.1147, -0.0216,  0.0824, -0.2227,  0.1101,  0.2807,  0.0139,  0.2451,
                       -0.2678,  0.1882],
                      [ 0.2211,  0.1689, -0.0283,  0.0555, -0.0697,  0.0540,  0.2401,  0.0699,
                        0.3083,  0.3003],
                      [-0.0309, -0.0716,  0.0863, -0.2755,  0.0915,  0.1958,  0.2541, -0.0222,
                       -0.1734,  0.2968],
                      [-0.0846, -0.2036,  0.0659, -0.2901,  0.1971,  0.0968,  0.1876,  0.0645,
                        0.1432, -0.2511],
                 

In [27]:
int8_layer.weight.dtype , int8_layer.bias.dtype

(torch.float32, torch.float32)

## sending to gpu

In [28]:
int8_layer = int8_layer.to(0) # Quantization happens here
int8_state = int8_layer.state_dict()
int8_state

OrderedDict([('weight',
              tensor([[  55,   54,   18, -127,   87,   76,  126,  -18,  -75,   64],
                      [  82,   58, -111,   -9,  -20,  -54,   12, -127,   38,  -83],
                      [ -62,  -18,  100,   85,   14, -102,  -41,  127,  -28, -117],
                      [ -52,  -10,   37, -101,   50,  127,    6,  111, -121,   85],
                      [  91,   70,  -12,   23,  -29,   22,   99,   29,  127,  124],
                      [ -13,  -31,   37, -118,   39,   84,  109,   -9,  -74,  127],
                      [ -37,  -89,   29, -127,   86,   42,   82,   28,   63, -110],
                      [-127, -109,  -97,   18,   11,   85,  -38,  -68,   34,  -36],
                      [-127,  -76,   -7,   13,    1,  124,   -2,  -21,   20,   64],
                      [ -93,  125, -127,  107,  -71,   11,  115,  -25, -103,  -54]],
                     device='cuda:0', dtype=torch.int8)),
             ('bias',
              tensor([-0.1067, -0.0213, -0.1924, -0.083

## this is quatanized layer now

8-bit quantization replaces all linear layers except for: 

◦ layers with tied (shared) weights

◦ the last layer in the model

◦ any layer named lm_head

## 4-Bit Quantization

## this has two datatypes

FP4 → regular 4-bit floating point

NF4 → Normalized Float4 (introduced in QLoRA paper)

# paper - qlora

"QLoRA is a finetuning method that quantizes a model to 4-bits and adds a set of low-rank adaptation (LoRA)
weights to the model and tuning them through the quantized weights. This method also introduces a new data
type, 4-bit NormalFloat (LinearNF4) in addition to the standard Float4 data type (LinearFP4). LinearNF4 is a
quantization data type for normally distributed data and can improve performance."

# Default config

In [37]:
bnb_4bit_quant_type="fp4"
bnb_4bit_use_double_quant=False
bnb_4bit_compute_dtype=torch.float32
bnb_4bit_quant_storage: torch.uint8

It’s perfectly fine to stick with all the default options, but you might still get some benefits from making your
own selections:

# Recommended config (best performance):Python

###  nf4 (normal float) quantization type can offer better performance.

 Double quantization can be used to quantize the constants from the first quantization (essentially, this is nested quantization). It is said to save an extra 0.4 bit per parameter.

torch.bfloat16 as the computation type (if your GPU supports it) or torch.float32 otherwise.

In [35]:
supported = torch.cuda.is_bf16_supported()   # True / False
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # ← use this
    bnb_4bit_use_double_quant=True,     # saves ~0.4 extra bit/parameter
    bnb_4bit_compute_dtype=torch.bfloat16 if supported else torch.float32
)

You can safely ignore the remaining argument, bnb_4bit_quant_storage, as it refers to the data type
internally used for storage.